【エクスポートデータ】 ・fiDF_adjusted.csv：共変量調整後の主成分の残差データ
。 ・AdjPval_FDR.csv：FDR補正済みの有意差（P値）リスト
。 ・CancerStatus_sampleID.csv：がんの有無に関連する調整済みデータセット
。 ・CancerProgress_sampleID.csv：がんの進行度に関連する調整済みデータセット
。

# Regression analysis

In [19]:
# conda install -c conda-forge scipy statsmodels

Retrieving notices: ...working... done
done
Solving environment: - 
The environment is inconsistent, please check the package plan carefully
The following packages are causing the inconsistency:

  - conda-forge/linux-64::libedit==3.1.20191231=he28a2e2_2
  - conda-forge/linux-64::libmamba==1.5.2=had39da4_0
  - conda-forge/linux-64::libmambapy==1.5.2=py311hf2555c7_0
  - conda-forge/linux-64::krb5==1.21.2=h659d440_0
  - conda-forge/linux-64::libcurl==8.4.0=hca28451_0
  - conda-forge/linux-64::conda==23.9.0=py311h38be061_0
  - conda-forge/linux-64::mamba==1.5.2=py311h3072747_0
  - conda-forge/noarch::r-evaluate==0.22=r43hc72bb7e_0
  - conda-forge/linux-64::r-warp==0.2.0=r43h57805ef_3
  - conda-forge/linux-64::r-bit==4.0.5=r43h57805ef_1
  - conda-forge/noarch::r-credentials==2.0.1=r43hc72bb7e_0
  - conda-forge/noarch::r-globals==0.16.2=r43hc72bb7e_1
  - conda-forge/noarch::r-pkgconfig==2.0.3=r43hc72bb7e_3
  - conda-forge/noarch::r-praise==1.0.0=r43hc72bb7e_1007
  - conda-forge/noarch::r-ha

In [1]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
#For Arial font
#!conda install -c conda-forge -y mscorefonts
##-> The below was also needed in matplotlib 3.4.2
#import shutil
#import matplotlib
#shutil.rmtree(matplotlib.get_cachedir())
import warnings
warnings.filterwarnings('ignore')
from IPython.display import display
import time
#For exporting .pdf file with editable text
import matplotlib
matplotlib.rcParams['pdf.fonttype']=42
matplotlib.rcParams['ps.fonttype']=42

import sys
from decimal import Decimal, ROUND_HALF_UP
import re

from sklearn.preprocessing import StandardScaler
import statsmodels.formula.api as smf
from statsmodels.stats import multitest as multi

!conda list

/opt/conda/lib/python3.11/site-packages/pandas/core/computation/expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/conda/lib/python3.11/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


ImportError: cannot import name '_lazywhere' from 'scipy._lib._util' (/opt/conda/lib/python3.11/site-packages/scipy/_lib/_util.py)

# Import data

## PCA data

In [3]:
fileDir = './Data/'
fileName = 'DataForRegression.csv'
# PancreasProject/Data/DataForRegression.csv
tempDF = pd.read_csv(fileDir + fileName)
tempDF = tempDF.set_index(tempDF.columns[0])
tempDF.index.name = None
display(tempDF)
display(tempDF.describe(include='all'))

pca_data = tempDF

,PC1 (19.9%),PC2 (7.0%),PC3 (3.8%),PC4 (3.4%),PC5 (2.4%),PC6 (2.2%),PC7 (1.7%),PC8 (1.4%),PC9 (1.3%),PC10 (1.1%),...,PC250 (0.1%),PC251 (0.1%),PC252 (0.1%),PC253 (0.1%),PC254 (0.1%),PC255 (0.1%),PC256 (0.1%),PC257 (0.1%),PC258 (0.1%),PC259 (0.1%)
PE_R478_001,-11.355640,2.211958,1.428997,0.986114,3.315245,4.351226,-1.813944,-2.447610,-1.905257,0.364327,...,-0.328205,1.198109,0.031885,1.475363,-1.099433,-1.210714,0.719848,1.552917,-0.160214,0.869287
PE_R478_002,28.766284,-2.515320,14.748306,7.497081,10.382521,9.502187,4.725005,27.820659,-11.860450,-5.800743,...,0.137454,-0.403069,0.056806,0.343244,-0.327806,0.151566,0.136003,-0.391108,0.298101,-0.054964
PE_R478_003,0.020907,-8.533393,-5.656833,-6.175430,2.122866,0.261548,-2.191429,-3.005755,0.388229,-0.717173,...,0.399104,-0.198771,-1.298470,-1.988539,-0.313614,-0.049286,-0.795734,0.704323,-0.138773,0.940349
PE_R478_004,1.280354,-0.511921,-0.282735,11.175303,-5.285951,2.919094,-4.031425,2.954687,-5.602279,-0.824243,...,0.186780,-1.379516,0.089493,0.620804,0.560463,-1.061113,0.420048,-0.203276,-0.792380,0.018937
PE_R478_005,47.165163,-4.037621,3.669131,13.197488,4.967250,-10.946485,3.679527,-4.453984,-2.679542,9.475449,...,0.620790,0.751181,-0.543947,0.253502,-0.044218,0.438867,0.551966,0.513971,0.012419,-0.122962
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
PE_R478_396,2.143146,4.313420,8.265516,6.833900,-11.311680,0.793727,-3.193656,2.985122,-7.668430,-2.905974,...,0.793516,0.446897,0.401149,-0.231640,-0.728203,0.270238,-0.830906,-0.726959,0.547466,0.355459
PE_R478_397,38.061665,11.087694,-4.575514,27.387240,-11.711320,-3.291643,2.291494,-1.995233,6.664463,-1.006462,...,1.302729,0.368778,0.418047,-0.090198,-0.159994,0.564599,-0.597246,0.055804,0.154272,-0.043898
PE_R478_398,-14.914629,-9.359351,-9.131304,6.261250,-2.796122,-0.106774,-4.011564,-0.376139,1.144779,2.105016,...,1.824800,-0.018572,-0.004333,-0.224347,-0.227958,0.755860,0.756803,0.358922,-0.225955,0.298425
PE_R478_399,-3.590776,3.370606,11.278625,1.408932,-2.785291,-4.212388,6.327413,4.066127,3.191082,0.282851,...,1.393078,-0.889175,1.445657,1.114790,1.940781,-0.019798,0.590128,-0.103966,1.768409,1.117973


,PC1 (19.9%),PC2 (7.0%),PC3 (3.8%),PC4 (3.4%),PC5 (2.4%),PC6 (2.2%),PC7 (1.7%),PC8 (1.4%),PC9 (1.3%),PC10 (1.1%),...,PC250 (0.1%),PC251 (0.1%),PC252 (0.1%),PC253 (0.1%),PC254 (0.1%),PC255 (0.1%),PC256 (0.1%),PC257 (0.1%),PC258 (0.1%),PC259 (0.1%)
count,3.980000e+02,3.980000e+02,3.980000e+02,3.980000e+02,3.980000e+02,3.980000e+02,3.980000e+02,3.980000e+02,3.980000e+02,3.980000e+02,...,3.980000e+02,3.980000e+02,3.980000e+02,3.980000e+02,3.980000e+02,3.980000e+02,3.980000e+02,398.000000,3.980000e+02,3.980000e+02
mean,5.712906e-16,7.141133e-17,1.785283e-17,2.499397e-16,-1.071170e-16,7.141133e-17,-3.570567e-17,1.785283e-17,-3.570567e-17,4.463208e-17,...,1.227382e-17,-4.909529e-17,4.574788e-17,-1.562123e-17,2.677925e-17,2.231604e-17,-3.124246e-17,0.000000,1.004222e-17,-8.926416e-18
std,1.437948e+01,8.497949e+00,6.259899e+00,5.904815e+00,4.963187e+00,4.757023e+00,4.177342e+00,3.839139e+00,3.735517e+00,3.447219e+00,...,8.242827e-01,8.227165e-01,8.200357e-01,8.162502e-01,8.087023e-01,8.063115e-01,7.945541e-01,0.793658,7.900064e-01,7.880754e-01
min,-2.711635e+01,-2.629658e+01,-1.668219e+01,-1.698651e+01,-1.455002e+01,-1.405542e+01,-1.096862e+01,-1.130787e+01,-1.186045e+01,-8.757062e+00,...,-2.471249e+00,-2.543121e+00,-2.137923e+00,-2.634246e+00,-2.330016e+00,-2.361457e+00,-2.471270e+00,-2.066688,-2.331782e+00,-2.581984e+00
25%,-9.358061e+00,-5.267991e+00,-4.268379e+00,-3.316688e+00,-3.367194e+00,-2.697410e+00,-2.966586e+00,-2.470633e+00,-2.424165e+00,-2.234652e+00,...,-4.701661e-01,-5.130118e-01,-5.406093e-01,-4.591096e-01,-5.088443e-01,-5.561077e-01,-4.538436e-01,-0.523605,-4.946316e-01,-5.302820e-01
50%,-2.902832e+00,3.796802e-02,-9.272805e-04,-3.387693e-01,-5.183615e-01,-2.951134e-02,-5.098784e-02,-2.486057e-01,-1.922131e-01,-1.219436e-01,...,3.143556e-02,3.826649e-02,5.398880e-02,7.067323e-02,-1.631901e-02,-3.712911e-02,7.984785e-03,-0.039082,-5.312299e-02,-1.260867e-02
75%,6.769484e+00,4.596052e+00,4.263626e+00,2.625137e+00,3.290777e+00,2.235187e+00,2.900097e+00,2.265234e+00,2.228297e+00,2.135437e+00,...,5.481562e-01,5.523407e-01,5.065511e-01,5.523370e-01,5.440846e-01,5.132725e-01,4.656954e-01,0.502430,5.070006e-01,5.096543e-01
max,6.505593e+01,2.788149e+01,2.469381e+01,3.355831e+01,1.780996e+01,2.436114e+01,1.447239e+01,2.782066e+01,1.558460e+01,1.175876e+01,...,2.631017e+00,2.599814e+00,2.542172e+00,2.817259e+00,3.164918e+00,2.438367e+00,2.533927e+00,2.134593,2.433389e+00,2.726829e+00


### delete % 

In [4]:
tempDF = pca_data
print("before: ")
display(tempDF)

tempDF.columns = [col.split()[0] for col in tempDF.columns]
print("After: ")
display(tempDF)
pca_data = tempDF

before: 


,PC1 (19.9%),PC2 (7.0%),PC3 (3.8%),PC4 (3.4%),PC5 (2.4%),PC6 (2.2%),PC7 (1.7%),PC8 (1.4%),PC9 (1.3%),PC10 (1.1%),...,PC250 (0.1%),PC251 (0.1%),PC252 (0.1%),PC253 (0.1%),PC254 (0.1%),PC255 (0.1%),PC256 (0.1%),PC257 (0.1%),PC258 (0.1%),PC259 (0.1%)
PE_R478_001,-11.355640,2.211958,1.428997,0.986114,3.315245,4.351226,-1.813944,-2.447610,-1.905257,0.364327,...,-0.328205,1.198109,0.031885,1.475363,-1.099433,-1.210714,0.719848,1.552917,-0.160214,0.869287
PE_R478_002,28.766284,-2.515320,14.748306,7.497081,10.382521,9.502187,4.725005,27.820659,-11.860450,-5.800743,...,0.137454,-0.403069,0.056806,0.343244,-0.327806,0.151566,0.136003,-0.391108,0.298101,-0.054964
PE_R478_003,0.020907,-8.533393,-5.656833,-6.175430,2.122866,0.261548,-2.191429,-3.005755,0.388229,-0.717173,...,0.399104,-0.198771,-1.298470,-1.988539,-0.313614,-0.049286,-0.795734,0.704323,-0.138773,0.940349
PE_R478_004,1.280354,-0.511921,-0.282735,11.175303,-5.285951,2.919094,-4.031425,2.954687,-5.602279,-0.824243,...,0.186780,-1.379516,0.089493,0.620804,0.560463,-1.061113,0.420048,-0.203276,-0.792380,0.018937
PE_R478_005,47.165163,-4.037621,3.669131,13.197488,4.967250,-10.946485,3.679527,-4.453984,-2.679542,9.475449,...,0.620790,0.751181,-0.543947,0.253502,-0.044218,0.438867,0.551966,0.513971,0.012419,-0.122962
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
PE_R478_396,2.143146,4.313420,8.265516,6.833900,-11.311680,0.793727,-3.193656,2.985122,-7.668430,-2.905974,...,0.793516,0.446897,0.401149,-0.231640,-0.728203,0.270238,-0.830906,-0.726959,0.547466,0.355459
PE_R478_397,38.061665,11.087694,-4.575514,27.387240,-11.711320,-3.291643,2.291494,-1.995233,6.664463,-1.006462,...,1.302729,0.368778,0.418047,-0.090198,-0.159994,0.564599,-0.597246,0.055804,0.154272,-0.043898
PE_R478_398,-14.914629,-9.359351,-9.131304,6.261250,-2.796122,-0.106774,-4.011564,-0.376139,1.144779,2.105016,...,1.824800,-0.018572,-0.004333,-0.224347,-0.227958,0.755860,0.756803,0.358922,-0.225955,0.298425
PE_R478_399,-3.590776,3.370606,11.278625,1.408932,-2.785291,-4.212388,6.327413,4.066127,3.191082,0.282851,...,1.393078,-0.889175,1.445657,1.114790,1.940781,-0.019798,0.590128,-0.103966,1.768409,1.117973


After: 


,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,...,PC250,PC251,PC252,PC253,PC254,PC255,PC256,PC257,PC258,PC259
PE_R478_001,-11.355640,2.211958,1.428997,0.986114,3.315245,4.351226,-1.813944,-2.447610,-1.905257,0.364327,...,-0.328205,1.198109,0.031885,1.475363,-1.099433,-1.210714,0.719848,1.552917,-0.160214,0.869287
PE_R478_002,28.766284,-2.515320,14.748306,7.497081,10.382521,9.502187,4.725005,27.820659,-11.860450,-5.800743,...,0.137454,-0.403069,0.056806,0.343244,-0.327806,0.151566,0.136003,-0.391108,0.298101,-0.054964
PE_R478_003,0.020907,-8.533393,-5.656833,-6.175430,2.122866,0.261548,-2.191429,-3.005755,0.388229,-0.717173,...,0.399104,-0.198771,-1.298470,-1.988539,-0.313614,-0.049286,-0.795734,0.704323,-0.138773,0.940349
PE_R478_004,1.280354,-0.511921,-0.282735,11.175303,-5.285951,2.919094,-4.031425,2.954687,-5.602279,-0.824243,...,0.186780,-1.379516,0.089493,0.620804,0.560463,-1.061113,0.420048,-0.203276,-0.792380,0.018937
PE_R478_005,47.165163,-4.037621,3.669131,13.197488,4.967250,-10.946485,3.679527,-4.453984,-2.679542,9.475449,...,0.620790,0.751181,-0.543947,0.253502,-0.044218,0.438867,0.551966,0.513971,0.012419,-0.122962
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
PE_R478_396,2.143146,4.313420,8.265516,6.833900,-11.311680,0.793727,-3.193656,2.985122,-7.668430,-2.905974,...,0.793516,0.446897,0.401149,-0.231640,-0.728203,0.270238,-0.830906,-0.726959,0.547466,0.355459
PE_R478_397,38.061665,11.087694,-4.575514,27.387240,-11.711320,-3.291643,2.291494,-1.995233,6.664463,-1.006462,...,1.302729,0.368778,0.418047,-0.090198,-0.159994,0.564599,-0.597246,0.055804,0.154272,-0.043898
PE_R478_398,-14.914629,-9.359351,-9.131304,6.261250,-2.796122,-0.106774,-4.011564,-0.376139,1.144779,2.105016,...,1.824800,-0.018572,-0.004333,-0.224347,-0.227958,0.755860,0.756803,0.358922,-0.225955,0.298425
PE_R478_399,-3.590776,3.370606,11.278625,1.408932,-2.785291,-4.212388,6.327413,4.066127,3.191082,0.282851,...,1.393078,-0.889175,1.445657,1.114790,1.940781,-0.019798,0.590128,-0.103966,1.768409,1.117973


## sample data

In [5]:
fileDir = "./Data/"
fileName = "pro_sample.csv"

# PancreasProject/Data/DataForRegression.csv
tempDF = pd.read_csv(fileDir + fileName)
tempDF = tempDF.set_index('Sample ID')
tempDF.index.name = None
display(tempDF)
display(tempDF.describe(include='all'))

pro_sample = tempDF

,progress,sex,fvage,CaCo_Status,icdo3t,icdo3m,smoker,py,drinker,quant_drink_gr,bmi,hx_dm,CA19_9,sIL2R
PE_R478_001,0,1,58,0,NaN,NaN,2,8,1,25,20,0,4.100,134.0
PE_R478_002,0,1,62,0,NaN,NaN,2,10,1,33,23,0,1.999,5500.0
PE_R478_003,0,1,68,0,NaN,NaN,1,24,3,0,22,0,2.700,259.0
PE_R478_004,0,2,70,0,NaN,NaN,3,0,3,0,26,0,3.800,208.0
PE_R478_005,440,1,59,1,C252,81403.0,3,0,3,0,29,0,1200.000,1181.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
PE_R478_396,410,1,63,1,C250,84533.0,1,27,3,0,26,1,1.999,263.0
PE_R478_397,410,2,64,1,C250,85003.0,3,0,1,12,22,0,1.999,257.0
PE_R478_398,0,1,38,0,NaN,NaN,2,13,1,4,29,0,2.100,257.0
PE_R478_399,410,2,47,1,C250,85003.0,3,0,1,2,21,0,16.900,216.0


,progress,sex,fvage,CaCo_Status,icdo3t,icdo3m,smoker,py,drinker,quant_drink_gr,bmi,hx_dm,CA19_9,sIL2R
count,398.000000,398.000000,398.000000,398.000000,200,200.000000,398.000000,398.000000,398.000000,398.000000,398.000000,398.000000,398.000000,398.000000
unique,NaN,NaN,NaN,NaN,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,NaN,NaN,NaN,NaN,C250,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,NaN,NaN,NaN,NaN,88,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,211.155779,1.429648,61.555276,0.502513,NaN,82320.250000,2.399497,14.170854,2.005025,13.610553,22.562814,0.168342,187.232035,339.077487
std,210.553238,0.495649,12.045703,0.500623,NaN,1500.386412,0.719431,22.662366,0.946943,24.659016,3.485099,0.374640,380.108959,380.197983
min,0.000000,1.000000,16.000000,0.000000,NaN,81403.000000,1.000000,0.000000,1.000000,0.000000,15.000000,0.000000,1.999000,99.990000
25%,0.000000,1.000000,56.000000,0.000000,NaN,81403.000000,2.000000,0.000000,1.000000,0.000000,20.000000,0.000000,4.000000,180.250000
50%,400.000000,1.000000,63.000000,1.000000,NaN,81403.000000,3.000000,0.000000,2.000000,2.000000,22.000000,0.000000,9.400000,256.500000
75%,410.000000,2.000000,70.000000,1.000000,NaN,84523.000000,3.000000,24.000000,3.000000,19.000000,25.000000,0.000000,68.000000,344.000000


## Merged data

In [6]:
merged_data = pd.concat([pca_data, pro_sample], axis = 1)
display(merged_data)

,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,...,icdo3t,icdo3m,smoker,py,drinker,quant_drink_gr,bmi,hx_dm,CA19_9,sIL2R
PE_R478_001,-11.355640,2.211958,1.428997,0.986114,3.315245,4.351226,-1.813944,-2.447610,-1.905257,0.364327,...,NaN,NaN,2,8,1,25,20,0,4.100,134.0
PE_R478_002,28.766284,-2.515320,14.748306,7.497081,10.382521,9.502187,4.725005,27.820659,-11.860450,-5.800743,...,NaN,NaN,2,10,1,33,23,0,1.999,5500.0
PE_R478_003,0.020907,-8.533393,-5.656833,-6.175430,2.122866,0.261548,-2.191429,-3.005755,0.388229,-0.717173,...,NaN,NaN,1,24,3,0,22,0,2.700,259.0
PE_R478_004,1.280354,-0.511921,-0.282735,11.175303,-5.285951,2.919094,-4.031425,2.954687,-5.602279,-0.824243,...,NaN,NaN,3,0,3,0,26,0,3.800,208.0
PE_R478_005,47.165163,-4.037621,3.669131,13.197488,4.967250,-10.946485,3.679527,-4.453984,-2.679542,9.475449,...,C252,81403.0,3,0,3,0,29,0,1200.000,1181.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
PE_R478_396,2.143146,4.313420,8.265516,6.833900,-11.311680,0.793727,-3.193656,2.985122,-7.668430,-2.905974,...,C250,84533.0,1,27,3,0,26,1,1.999,263.0
PE_R478_397,38.061665,11.087694,-4.575514,27.387240,-11.711320,-3.291643,2.291494,-1.995233,6.664463,-1.006462,...,C250,85003.0,3,0,1,12,22,0,1.999,257.0
PE_R478_398,-14.914629,-9.359351,-9.131304,6.261250,-2.796122,-0.106774,-4.011564,-0.376139,1.144779,2.105016,...,NaN,NaN,2,13,1,4,29,0,2.100,257.0
PE_R478_399,-3.590776,3.370606,11.278625,1.408932,-2.785291,-4.212388,6.327413,4.066127,3.191082,0.282851,...,C250,85003.0,3,0,1,2,21,0,16.900,216.0


In [7]:
tempDF = merged_data

#Remove unnecessary columns
tempL = ['py', 'bmi', 'quant_drink_gr', 'fvage','sex','hx_dm']
tempL2 = ['sex','hx_dm']
tempL3 = ['py', 'bmi', 'quant_drink_gr', 'fvage']

tempDF2 = tempDF[tempL]

display(tempDF2)
display(tempDF[tempL2].astype('category').describe())
display(tempDF[tempL3].astype('int').describe())


varDF = tempDF2

,py,bmi,quant_drink_gr,fvage,sex,hx_dm
PE_R478_001,8,20,25,58,1,0
PE_R478_002,10,23,33,62,1,0
PE_R478_003,24,22,0,68,1,0
PE_R478_004,0,26,0,70,2,0
PE_R478_005,0,29,0,59,1,0
...,...,...,...,...,...,...
PE_R478_396,27,26,0,63,1,1
PE_R478_397,0,22,12,64,2,0
PE_R478_398,13,29,4,38,1,0
PE_R478_399,0,21,2,47,2,0


,sex,hx_dm
count,398,398
unique,2,2
top,1,0
freq,227,331


,py,bmi,quant_drink_gr,fvage
count,398.000000,398.000000,398.000000,398.000000
mean,14.170854,22.562814,13.610553,61.555276
std,22.662366,3.485099,24.659016,12.045703
min,0.000000,15.000000,0.000000,16.000000
25%,0.000000,20.000000,0.000000,56.000000
50%,0.000000,22.000000,2.000000,63.000000
75%,24.000000,25.000000,19.000000,70.000000
max,168.000000,34.000000,215.000000,88.000000


## Preparation

In [8]:
tempDF = pro_sample
tempDF1 = tempDF[['CaCo_Status', 'progress']].copy()
tempDF1 = tempDF1.rename(columns = {'CaCo_Status': 'CancerStatus','progress': 'CancerProgress'})
display(tempDF1)
targetDF = tempDF1

,CancerStatus,CancerProgress
PE_R478_001,0,0
PE_R478_002,0,0
PE_R478_003,0,0
PE_R478_004,0,0
PE_R478_005,1,440
...,...,...
PE_R478_396,1,410
PE_R478_397,1,410
PE_R478_398,0,0
PE_R478_399,1,410


In [9]:
tempDF = varDF.copy()
tempDF2 = pca_data.copy()
nPCs = 12

tempDF3 = tempDF2.iloc[:, :nPCs]
tempDF = pd.merge(tempDF3, tempDF, left_index=True, right_index=True, how='left')
tempDF = pd.merge(tempDF, targetDF, left_index=True, right_index=True, how='right')

display(tempDF)

varDF = tempDF

,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,PC11,PC12,py,bmi,quant_drink_gr,fvage,sex,hx_dm,CancerStatus,CancerProgress
PE_R478_001,-11.355640,2.211958,1.428997,0.986114,3.315245,4.351226,-1.813944,-2.447610,-1.905257,0.364327,1.629236,-2.879106,8,20,25,58,1,0,0,0
PE_R478_002,28.766284,-2.515320,14.748306,7.497081,10.382521,9.502187,4.725005,27.820659,-11.860450,-5.800743,18.474892,1.146068,10,23,33,62,1,0,0,0
PE_R478_003,0.020907,-8.533393,-5.656833,-6.175430,2.122866,0.261548,-2.191429,-3.005755,0.388229,-0.717173,0.162424,-0.540679,24,22,0,68,1,0,0,0
PE_R478_004,1.280354,-0.511921,-0.282735,11.175303,-5.285951,2.919094,-4.031425,2.954687,-5.602279,-0.824243,2.481981,4.804320,0,26,0,70,2,0,0,0
PE_R478_005,47.165163,-4.037621,3.669131,13.197488,4.967250,-10.946485,3.679527,-4.453984,-2.679542,9.475449,-3.626041,11.273339,0,29,0,59,1,0,1,440
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
PE_R478_396,2.143146,4.313420,8.265516,6.833900,-11.311680,0.793727,-3.193656,2.985122,-7.668430,-2.905974,-4.713580,3.896034,27,26,0,63,1,1,1,410
PE_R478_397,38.061665,11.087694,-4.575514,27.387240,-11.711320,-3.291643,2.291494,-1.995233,6.664463,-1.006462,3.172848,-7.012953,0,22,12,64,2,0,1,410
PE_R478_398,-14.914629,-9.359351,-9.131304,6.261250,-2.796122,-0.106774,-4.011564,-0.376139,1.144779,2.105016,2.228158,-1.960262,13,29,4,38,1,0,0,0
PE_R478_399,-3.590776,3.370606,11.278625,1.408932,-2.785291,-4.212388,6.327413,4.066127,3.191082,0.282851,-1.018348,-0.511177,0,21,2,47,2,0,1,410


# Regression analysis

In [10]:
#修正版
tempDF1 = pca_data
tempL = ['py', 'bmi', 'quant_drink_gr', 'fvage','sex','hx_dm']
tempDF2 = varDF[tempL]
tempL1 = ['CancerStatus', 'CancerProgress']#Target dependent variables
formula1 = 'PC ~ C(CancerVariable) + fvage + py + quant_drink_gr + bmi + C(sex) + C(hx_dm)'#Full model
formula2 = 'PC ~ fvage + py + quant_drink_gr + bmi + C(sex) + C(hx_dm)' #Covariate model to get residuals
statDF = pd.DataFrame()
fiDF_adjusted = pd.DataFrame(index=tempDF1.index)
t_start = time.time()


for module in tempDF1.columns:
    for variable in tempL1:
        #Prepare target DF
        tempS = tempDF1[module]
        tempS = tempS.to_frame(name='PC')
        tempDF = pd.merge(tempS, tempDF2, left_index=True, right_index=True, how='left')
        # tempDF = tempDF.rename(columns={module:'bmi', variable:'FrailtyIndex'})
        tempDF['CancerVariable'] = varDF[variable]
        
        #Standardization of continuous variables
        tempDF3 = tempDF.select_dtypes(include=np.number)
        scaler = StandardScaler(copy=True, with_mean=True, with_std=True)
        tempA = scaler.fit_transform(tempDF3)#Column direction
        tempDF3 = pd.DataFrame(data=tempA, index=tempDF3.index, columns=tempDF3.columns)
        tempDF4 = tempDF.select_dtypes(exclude=np.number)
        tempDF3 = pd.merge(tempDF3, tempDF4, left_index=True, right_index=True, how='left')
        
        #Fit the full model
        model = smf.ols(formula1, data=tempDF3).fit()
        print(formula1.replace('CancerVariable', variable)+'('+module+')')
        display(model.summary())
        print('')

        
        #Save
        tempS1 = model.params
        tempS1.name = 'bCoef'
        tempS2 = model.bse
        tempS2.name = 'bCoefSE'
        tempS3 = model.tvalues
        tempS3.name = 'tStat'
        tempS4 = model.pvalues
        tempS4.name = 'Pval'
        tempS5 = model.conf_int(alpha=0.05).iloc[:, 0]
        tempS5.name = '95CIlow'
        tempS6 = model.conf_int(alpha=0.05).iloc[:, 1]
        tempS6.name = '95CIhigh'
        tempDF = pd.concat([tempS1, tempS2, tempS3, tempS4, tempS5, tempS6], axis=1)
        tempDF.index.name = 'Variable'
        tempDF = tempDF.reset_index()
        tempDF['ModuleID'] = module
        tempDF['FItype'] = variable
        statDF = pd.concat([statDF, tempDF], axis=0)
        
        #Calculate adjusted values
        model = smf.ols(formula2, data=tempDF3).fit()
        tempS = model.resid + tempDF3['CancerVariable'].mean()

        ##Transform back to the original scale
        tempDF = tempDF3.select_dtypes(include=np.number)
        tempDF['CancerVariable'] = tempS.to_numpy()#Replace to prepare the other dummy columns
        tempA = scaler.inverse_transform(tempDF)
        tempDF = pd.DataFrame(data=tempA, index=tempDF.index, columns=tempDF.columns)
        tempS = tempDF['CancerVariable'].copy()
        tempS.name = module+'-'+variable
        fiDF_adjusted = pd.merge(fiDF_adjusted, tempS, left_index=True, right_index=True, how='left')
    print('----------------------------------------------------------------------------------------------------\n')
t_elapsed = time.time() - t_start
print(len(tempDF1.columns)*len(tempL1), 'OLS regressions (',
      len(tempDF1.columns),'modules x',len(tempL1),'variables ):',
      round(t_elapsed//60), 'min', round(t_elapsed%60, 1), 'sec')

#Clean
statDF = statDF.set_index(['ModuleID', 'FItype'])
##P-value adjustment by using Benjamini–Hochberg method
statDF['AdjPval'] = multi.multipletests(statDF['Pval'], alpha=0.05, method='fdr_bh', is_sorted=False, returnsorted=False)[1]

display(statDF)
display(fiDF_adjusted)


NameError: name 'smf' is not defined

In [ ]:
#修正版
tempDF1 = pca_data
tempL = ['py', 'bmi', 'quant_drink_gr', 'fvage','sex','hx_dm']
tempDF2 = varDF[tempL]
tempL1 = ['CancerStatus', 'CancerProgress']#Target dependent variables
formula1 = 'PC ~ CancerVariable + fvage + py + quant_drink_gr + bmi + C(sex) + C(hx_dm)'#Full model
formula2 = 'PC ~ fvage + py + quant_drink_gr + bmi + C(sex) + C(hx_dm)' #Covariate model to get residuals
statDF = pd.DataFrame()
fiDF_adjusted = pd.DataFrame(index=tempDF1.index)
t_start = time.time()


for module in tempDF1.columns:
    for variable in tempL1:
        #Prepare target DF
        tempS = tempDF1[module]
        tempS = tempS.to_frame(name='PC')
        tempDF = pd.merge(tempS, tempDF2, left_index=True, right_index=True, how='left')
        # tempDF = tempDF.rename(columns={module:'bmi', variable:'FrailtyIndex'})
        tempDF['CancerVariable'] = varDF[variable]
        
        #Standardization of continuous variables
        tempDF3 = tempDF.select_dtypes(include=np.number)
        scaler = StandardScaler(copy=True, with_mean=True, with_std=True)
        tempA = scaler.fit_transform(tempDF3)#Column direction
        tempDF3 = pd.DataFrame(data=tempA, index=tempDF3.index, columns=tempDF3.columns)
        tempDF4 = tempDF.select_dtypes(exclude=np.number)
        tempDF3 = pd.merge(tempDF3, tempDF4, left_index=True, right_index=True, how='left')
        
        #Fit the full model
        model = smf.ols(formula1, data=tempDF3).fit()
        print(formula1.replace('CancerVariable', variable)+'('+module+')')
        display(model.summary())
        print('')

        
        #Save
        tempS1 = model.params
        tempS1.name = 'bCoef'
        tempS2 = model.bse
        tempS2.name = 'bCoefSE'
        tempS3 = model.tvalues
        tempS3.name = 'tStat'
        tempS4 = model.pvalues
        tempS4.name = 'Pval'
        tempS5 = model.conf_int(alpha=0.05).iloc[:, 0]
        tempS5.name = '95CIlow'
        tempS6 = model.conf_int(alpha=0.05).iloc[:, 1]
        tempS6.name = '95CIhigh'
        tempDF = pd.concat([tempS1, tempS2, tempS3, tempS4, tempS5, tempS6], axis=1)
        tempDF.index.name = 'Variable'
        tempDF = tempDF.reset_index()
        tempDF['ModuleID'] = module
        tempDF['FItype'] = variable
        statDF = pd.concat([statDF, tempDF], axis=0)
        
        #Calculate adjusted values
        model = smf.ols(formula2, data=tempDF3).fit()
        tempS = model.resid + tempDF3['CancerVariable'].mean()

        ##Transform back to the original scale
        tempDF = tempDF3.select_dtypes(include=np.number)
        tempDF['CancerVariable'] = tempS.to_numpy()#Replace to prepare the other dummy columns
        tempA = scaler.inverse_transform(tempDF)
        tempDF = pd.DataFrame(data=tempA, index=tempDF.index, columns=tempDF.columns)
        tempS = tempDF['CancerVariable'].copy()
        tempS.name = module+'-'+variable
        fiDF_adjusted = pd.merge(fiDF_adjusted, tempS, left_index=True, right_index=True, how='left')
    print('----------------------------------------------------------------------------------------------------\n')
t_elapsed = time.time() - t_start
print(len(tempDF1.columns)*len(tempL1), 'OLS regressions (',
      len(tempDF1.columns),'modules x',len(tempL1),'variables ):',
      round(t_elapsed//60), 'min', round(t_elapsed%60, 1), 'sec')

#Clean
statDF = statDF.set_index(['ModuleID', 'FItype'])
##P-value adjustment by using Benjamini–Hochberg method
statDF['AdjPval'] = multi.multipletests(statDF['Pval'], alpha=0.05, method='fdr_bh', is_sorted=False, returnsorted=False)[1]

display(statDF)
display(fiDF_adjusted)


### Check adjPval

In [11]:
display(statDF)
display(fiDF_adjusted)

tempDF = statDF[statDF['Variable']=='CancerVariable']
display(tempDF)
print(tempDF['AdjPval'].describe())

""


""
PE_R478_001
PE_R478_002
PE_R478_003
PE_R478_004
PE_R478_005
...
PE_R478_396
PE_R478_397
PE_R478_398
PE_R478_399


KeyError: 'Variable'

In [ ]:
statusDF = fiDF_adjusted[[col for col in fiDF_adjusted.columns if 'CancerStatus' in col]]

progressDF = fiDF_adjusted[[col for col in fiDF_adjusted.columns if 'CancerProgress' in col]]

display(statusDF)
display(progressDF)

## Visualization across modules

In [ ]:
#Visualize results across WGCNA modules
variable = 'CancerVariable'
tempDF = statDF.loc[statDF['Variable']==variable]
tempD1 = {'CancerStatus':'tab:blue', 'CancerProgress':'tab:red'}#Target dependent variables
tempD2 = {'CancerStatus':'Status of cancer', 'CancerProgress':'Progress of cancer'}
figtitle = 'PC ~ CancerVariable + BMI + Sex + Age + Pack years + Daily alcohol intake(g) + History of diabetes'

#P-value adjustment by using Benjamini–Hochberg method
tempDF['AdjPval'] = multi.multipletests(tempDF['Pval'], alpha=0.05, method='fdr_bh',
                                        is_sorted=False, returnsorted=False)[1]

#Calculate CI range
tempDF['CIrange'] = (tempDF['95CIhigh'] - tempDF['95CIlow']) / 2

#Significance annotation
tempL = []
for row_i in range(len(tempDF)):
    adjpval = tempDF['AdjPval'].iloc[row_i]
    if adjpval<0.001:
        tempL.append('***')
    elif adjpval<0.01:
        tempL.append('**')
    elif adjpval<0.05:
        tempL.append('*')
    else:
        tempL.append('n.s.')
tempDF['Signif'] = tempL
# 有意なPC（多重検定補正後 p < 0.05）のみ残す
tempDF = tempDF.loc[tempDF['AdjPval'] < 0.05].copy()

# 有意なPCがない場合の確認
if tempDF.empty:
    print("有意なPCはありません。")
#Sort
tempDF = tempDF.sort_values(by=['bCoef'], ascending=False)
tempDF = tempDF.reset_index()#Note that the row number index is used later
tempA = tempDF['ModuleID'].unique()
tempDF['FItypelabel'] = tempDF['FItype'].map(tempD2)

#Visualize beta-coefficients with forest plot
##-> Manual plot to add significance signs with multiple levels
sns.set(style='ticks', font='DejaVu Sans', context='talk')
fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(9, len(tempA)*len(tempD1)/5))
##Add points with multiple levels
sns.pointplot(data=tempDF, x='bCoef', y='ModuleID', order=tempA,
              hue='FItype', hue_order=tempD1.keys(), palette=tempD1,
              dodge=0.6, join=False, ci=None, markers='o', scale=0.6, ax=ax)
##Get coordinate of each point
xcoordL = []
ycoordL = []
for line in ax.lines:
    xdata = line.get_xdata()
    ydata = line.get_ydata()
    for x, y in zip(xdata, ydata):
        xcoordL.append(x)
        ycoordL.append(y)

##Add errorbars and annotations
for point_i in range(len(tempDF)):
    #Be careful about order of ax.collections: tempA -> tempD1.keys()
    module = tempA[point_i%len(tempA)]
    fitype = list(tempD1.keys())[point_i//len(tempA)]
    row_i = tempDF.loc[(tempDF['ModuleID']==module)&(tempDF['FItype']==fitype)].index.tolist()[0]
    #Add errorbars manually    
    ax.errorbar(x=xcoordL[point_i], y=ycoordL[point_i], xerr=tempDF['CIrange'].iloc[row_i],
                fmt='', ecolor=tempD1[fitype], elinewidth=4, capsize=0, capthick=4,
                linestyle='', zorder=1)
    #Add significance
    signif = tempDF['Signif'].iloc[row_i]
    if signif!='n.s.':
        text_offset = 5
        if xcoordL[point_i] > 0:
            xoffset = +text_offset
            halign = 'left'
            xcoord = xcoordL[point_i] + tempDF['CIrange'].iloc[row_i]
        else:
            xoffset = -text_offset
            halign = 'right'
            xcoord = xcoordL[point_i] - tempDF['CIrange'].iloc[row_i]
        ycoord = ycoordL[point_i]
        yoffset = -4#Because asterisk looks upper shift
        ax.annotate(signif, xy=(xcoord, ycoord), xytext=(xoffset, yoffset), textcoords='offset points',
                    horizontalalignment=halign, verticalalignment='center',
                    fontsize='medium', color=tempD1[fitype], zorder=2)
##Add reference line
ax.axvline(x=0, **{'linestyle':'--', 'color':'k', 'zorder':0})
##Axis settings
sns.despine()
plt.setp(ax, ylim=(len(tempA)-0.5, -0.5))#Invert and eliminate unnecessary spaces
plt.setp(ax, xlabel='Adjusted '+r'$\beta$'+'-coefficient ('+r'$Z$'+'-score)', ylabel='module')
##Add shading (just for aesthetic purpose)
for ycoord in range(len(tempA)):
    if ycoord%2 == 0:
        ax.axhspan(ymin=ycoord-0.5, ymax=ycoord+0.5, facecolor='k', alpha=0.2, zorder=0)
##Add legend
ax.legend(title='FI type', loc='lower right', handlelength=1.0, handletextpad=0.5)
fig.suptitle(figtitle, size='medium',
             verticalalignment='bottom', horizontalalignment='center', y=0.9)
plt.show()


In [ ]:


fig.savefig("./ExportData/pcaplot_all.svg",dpi = 300,
    bbox_inches="tight",transparent=True)
AdjPvalDF = tempDF

### Save adjusted Pvalue

In [12]:
display(fiDF_adjusted)

""
PE_R478_001
PE_R478_002
PE_R478_003
PE_R478_004
PE_R478_005
...
PE_R478_396
PE_R478_397
PE_R478_398
PE_R478_399


In [13]:
tempDF = fiDF_adjusted

fileDir = './Data/'
fileName = 'fiDF_adjusted.csv'
tempDF.to_csv(fileDir + fileName, index = True)

## Visualization 2

### Check adjPval

In [14]:
display(statDF)
display(fiDF_adjusted)

tempDF = statDF[statDF['Variable']=='CancerVariable']
display(tempDF)
print(tempDF['AdjPval'].describe())
print('PC21:',tempDF.loc['PC21','Pval'])
print('PC1:',tempDF.loc['PC1','Pval'])
print('PC4:',tempDF.loc['PC4','Pval'])
print('PC7:',tempDF.loc['PC7','Pval'])
print('PC5:',tempDF.loc['PC5','Pval'])
print('PC10:',tempDF.loc['PC10','Pval'])
print('PC6:',tempDF.loc['PC6','Pval'])

""


""
PE_R478_001
PE_R478_002
PE_R478_003
PE_R478_004
PE_R478_005
...
PE_R478_396
PE_R478_397
PE_R478_398
PE_R478_399


KeyError: 'Variable'

### save Adjusted P value(FDR)

In [15]:
tempDF = statDF.loc[statDF['Variable']=='CancerVariable']

display(tempDF)

KeyError: 'Variable'

In [16]:
display(AdjPvalDF)

NameError: name 'AdjPvalDF' is not defined

In [ ]:
tempDF = AdjPvalDF

fileDir = './Data/'
fileName = 'AdjPval_FDR.csv'
tempDF.to_csv(fileDir + fileName, index = True)

# Save

## cancer status data

In [17]:
tempDF = statusDF

fileDir = './Data/'
fileName = 'CancerStatus_sampleID.csv'
tempDF.to_csv(fileDir + fileName, index = True)

NameError: name 'statusDF' is not defined

## cancer progress data

In [ ]:
tempDF = progressDF

fileDir = './Data/'
fileName = 'CancerProgress_sampleID.csv'
tempDF.to_csv(fileDir + fileName, index = True)